# Comprehensive ETL Pipeline
**Data Collection and Harmonization**

This notebook creates a comprehensive ETL pipeline that collects data from multiple sources and loads it into a DuckDB database with a harmonized format: **ORIGIN - SERIES - DATE - VALUE**.

## Data Sources:
- FRED API (Federal Reserve Economic Data)
- Yahoo Finance (Stocks, ETFs, Commodities)
- Danmarks Statistik (Danish Statistics)
- FinansDanmark (Danish Financial Industry)
- Flat stats files from FinansDanmarks boligstatistik (Housing Statistics)

The pipeline implements delta loading to only populate the database with new values.

## Setup and Libraries

In [ ]:
setwd("~/learning/money-printer-go-brrr/datacollection")

In [ ]:
# Set options
knitr::opts_chunk$set(echo = TRUE, message = FALSE, warning = FALSE)
options(scipen = 999)

In [ ]:
# Load custom theme and plotting functions
source("~/renv_start.R")
source("~/theme_money_printer_go_brrr.R")
source("~/save_plot.R")
source("~/save_table.R")
source("~/powerpoint_with_annotations.R")
source("~/nnedl_connection.R")

# Logger
source("~/logger.R")
# set_loginfo_error_handler()

# Set up OpenAI API key for PowerPoint annotation
dotenv::load_dot_env("~/.env")

# Set global options
options(scipen = 999)  # Disable scientific notation
set.seed(42)  # For reproducibility

In [ ]:
# Define required packages
required_packages <- c(
  # Data manipulation
  "dplyr",
  "tidyr",
  "lubridate",
  "data.table",
  "ISOweek",        # For proper ISO week date handling
  
  # Database
  "DBI",
  "duckdb",
  
  # API connections
  "fredr",          # FRED API
  "quantmod",       # Yahoo Finance
  # "dkstat",         # Danmarks Statistik
  "httr",           # HTTP requests
  
  # File handling
  "openxlsx",
  "rvest",          # Web scraping
  "xml2",
  
  # Utilities
  "glue",
  "progress"
)

# CRITICAL: Update xml2 first if needed (rvest requires xml2 >= 1.4.0)
# Check if xml2 is already loaded with old version
if ("xml2" %in% loadedNamespaces()) {
  xml2_version <- as.character(packageVersion("xml2"))
  if (package_version(xml2_version) < "1.4.0") {
    cat("Detected old xml2 version:", xml2_version, "\n")
    cat("Please restart the R kernel and run this cell again.\n")
    cat("Or run: install.packages('xml2', dependencies = TRUE) and restart kernel.\n")
    stop("xml2 version >= 1.4.0 required. Please restart R kernel after updating.")
  }
} else {
  # xml2 not loaded yet - ensure latest version is installed
  if ("xml2" %in% installed.packages()[,"Package"]) {
    xml2_version <- as.character(packageVersion("xml2"))
    if (package_version(xml2_version) < "1.4.0") {
      cat("Updating xml2 from", xml2_version, "to latest version...\n")
      install.packages("xml2", dependencies = TRUE)
    }
  }
}

# Install and load packages using loop
for (pkg in required_packages) {
  if (!require(pkg, character.only = TRUE, quietly = TRUE)) {
    install.packages(pkg)
    library(pkg, character.only = TRUE)
  }
}

# Install dkstat from ropengov if not available
if (!"dkstat" %in% installed.packages()[,"Package"]) {
  install.packages(
  "dkstat",
    repos = c(
      ropengov = "https://ropengov.r-universe.dev",
      getOption("repos")
    )
)
}
library(dkstat)

print("All packages loaded successfully!")

## Database Setup

In [ ]:
# Database configuration
DB_PATH <- "data/financial_data.duckdb"

# Create data directory if it doesn't exist
if (!dir.exists("data")) {
  dir.create("data", recursive = TRUE)
  print("Created data directory")
}

# Check if directory was created successfully
if (!dir.exists("data")) {
  stop("Failed to create data directory")
}

# Debug: Check directory permissions and existence
print(glue("Data directory exists: {dir.exists('data')}"))
print(glue("Data directory writable: {file.access('data', mode = 2) == 0}"))
print(glue("Current working directory: {getwd()}"))
print(glue("Full data path: {normalizePath('data', mustWork = FALSE)}"))

# Check if 'con' already exists and disconnect if so
if (exists("con")) {
  tryCatch({
    if (dbIsValid(con)) {
      print("Closing existing database connection...")
      dbDisconnect(con, shutdown = TRUE)
    }
  }, error = function(e) {
    # Ignore errors from already disconnected connections
  })
}

# Shutdown any existing DuckDB driver instances to release locks
tryCatch(duckdb::duckdb_shutdown(duckdb::duckdb()), error = function(e) {})
gc()

# Remove stale lock/WAL files if no process is using the database
wal_file <- paste0(DB_PATH, ".wal")
if (file.exists(wal_file)) {
  cat("Removing stale WAL file...\n")
  file.remove(wal_file)
}

# Connect to DuckDB with automatic lock recovery
con <- tryCatch({
  dbConnect(duckdb::duckdb(), dbdir = DB_PATH)
}, error = function(e) {
  if (grepl("lock|locked|busy", e$message, ignore.case = TRUE)) {
    cat("\n=== DATABASE LOCK ERROR - attempting automatic recovery ===\n")
    
    # Remove WAL/tmp files
    lock_files <- Sys.glob(paste0(DB_PATH, ".*"))
    if (length(lock_files) > 0) {
      cat("Removing lock files:", paste(lock_files, collapse = ", "), "\n")
      file.remove(lock_files)
    }
    
    # Clear stale NFS locks by copy-replace
    if (file.exists(DB_PATH)) {
      backup <- paste0(DB_PATH, ".unlock_backup")
      cat("Clearing stale file lock via copy-replace...\n")
      file.copy(DB_PATH, backup, overwrite = TRUE)
      file.remove(DB_PATH)
      file.rename(backup, DB_PATH)
    }
    
    # Retry connection after cleanup
    tryCatch({
      dbConnect(duckdb::duckdb(), dbdir = DB_PATH)
    }, error = function(e2) {
      stop(paste("Cannot open database after lock recovery.",
                 "Please restart the R kernel.",
                 "Original error:", e$message))
    })
  } else {
    stop(e$message)
  }
})
# Now check if file exists
if (file.exists(DB_PATH)) {
  print(glue("Database file created successfully at: {DB_PATH}"))
  print(glue("Database file size: {file.size(DB_PATH)} bytes"))
} else {
  print("Warning: Database file still not found - trying alternative approach")
  
  # Close connection and try with absolute path
  dbDisconnect(con, shutdown = TRUE)
  
  # Use absolute path
  abs_path <- file.path(getwd(), "data", "financial_data.duckdb")
  print(glue("Trying absolute path: {abs_path}"))
  
  con <- dbConnect(duckdb::duckdb(), dbdir = abs_path)
  DB_PATH <- abs_path
  
  # Force write again
  dbExecute(con, "CREATE TABLE IF NOT EXISTS test_table (id INTEGER)")
  dbExecute(con, "DROP TABLE IF EXISTS test_table")
  
  if (file.exists(DB_PATH)) {
    print(glue("Database file created with absolute path: {DB_PATH}"))
  } else {
    print("Error: Unable to create database file - using in-memory database")
  }
}

# Create harmonized table with schema: ORIGIN - SERIES - DATE - VALUE
dbExecute(con, "
  CREATE TABLE IF NOT EXISTS financial_data (
    origin VARCHAR NOT NULL,
    series VARCHAR NOT NULL,
    date DATE NOT NULL,
    value DOUBLE,
    PRIMARY KEY (origin, series, date)
  )
")

# Create index for better performance
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_origin_series_date ON financial_data (origin, series, date)")
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_date ON financial_data (date)")

# Verify table was created
tables <- dbListTables(con)
if ("financial_data" %in% tables) {
  print("Table 'financial_data' created successfully")
} else {
  stop("Failed to create financial_data table")
}

print("Database initialized successfully!")
print(glue("Final database path: {DB_PATH}"))
print(glue("Database file exists: {file.exists(DB_PATH)}"))
if (file.exists(DB_PATH)) {
  print(glue("Database file size: {file.size(DB_PATH)} bytes"))
}

## Utility Functions

In [ ]:
# Function to get the latest date for a specific origin and series
get_latest_date <- function(con, origin, series = NULL) {
  if (is.null(series)) {
    query <- glue("SELECT MAX(date) as max_date FROM financial_data WHERE origin = '{origin}'")
  } else {
    query <- glue("SELECT MAX(date) as max_date FROM financial_data WHERE origin = '{origin}' AND series = '{series}'")
  }
  
  result <- dbGetQuery(con, query)
  if (is.na(result$max_date[1])) {
    return(as.Date("1900-01-01"))  # Return very old date if no data exists
  }
  return(as.Date(result$max_date[1]))
}

# Function to expand time series to daily frequency with LOCF
expand_to_daily_locf <- function(data) {
  if (nrow(data) == 0) return(data)
  
  # Get date range
  min_date <- min(data$date)
  max_date <- max(data$date)
  
  # Create complete daily date sequence
  all_dates <- data.frame(
    date = seq(min_date, max_date, by = "day")
  )
  
  # Merge with actual data
  expanded <- merge(all_dates, data, by = "date", all.x = TRUE)
  
  # Apply LOCF (Last Observation Carried Forward)
  expanded$value <- zoo::na.locf(expanded$value, na.rm = FALSE)
  
  # Remove leading NAs (before first observation)
  first_valid <- which(!is.na(expanded$value))[1]
  if (!is.na(first_valid) && first_valid > 1) {
    expanded <- expanded[first_valid:nrow(expanded), ]
  }
  
  # Remove any remaining NAs
  expanded <- expanded[!is.na(expanded$value), ]
  
  return(expanded)
}

# Function to insert data with delta loading (now applies LOCF by default)
insert_data <- function(con, data, origin, debug = FALSE, apply_locf = TRUE) {
  if (nrow(data) == 0) {
    print(glue("No new data to insert for origin: {origin}"))
    return(0)
  }
  
  if (debug) {
    print(glue("DEBUG insert_data - Input rows: {nrow(data)}"))
    print("DEBUG - Input column names:")
    print(names(data))
    print("DEBUG - First 3 rows of input:")
    print(head(data, 3))
  }
  
  # Ensure proper column names and types
  data <- data %>%
    mutate(
      origin = origin,
      date = as.Date(date),
      value = as.numeric(value)
    ) %>%
    select(origin, series, date, value) %>%
    filter(!is.na(value))
  
  # Apply LOCF expansion if requested (expands each series to daily)
  if (apply_locf && nrow(data) > 0) {
    # Group by series and apply LOCF to each
    unique_series <- unique(data$series)
    expanded_list <- list()
    
    for (s in unique_series) {
      series_data <- data[data$series == s, ]
      expanded <- expand_to_daily_locf(series_data)
      expanded$origin <- origin
      expanded$series <- s
      expanded_list[[s]] <- expanded
    }
    
    data <- do.call(rbind, expanded_list)
    row.names(data) <- NULL
    
    if (debug) {
      print(glue("DEBUG - After LOCF expansion: {nrow(data)} rows"))
    }
  }
  
  if (debug) {
    print(glue("DEBUG - After column selection and filtering, rows: {nrow(data)}"))
    print("DEBUG - First 3 rows after transformation:")
    print(head(data, 3))
  }
  
  if (nrow(data) == 0) {
    print(glue("No valid data to insert for origin: {origin}"))
    return(0)
  }
  
  # Get the series name from the data (assumes all rows have the same series)
  series_name <- unique(data$series)[1]
  
  # Remove duplicates that already exist in the database
  # Create a temporary table to check for existing records
  temp_table_name <- paste0("temp_", origin, "_", format(Sys.time(), "%Y%m%d_%H%M%S"))
  temp_table_name <- gsub("[^A-Za-z0-9_]", "_", temp_table_name)
  
  tryCatch({
    # Write data to temporary table
    dbWriteTable(con, temp_table_name, data, temporary = TRUE)
    
    if (debug) {
      # Check how many records are in the temp table
      temp_count <- dbGetQuery(con, glue("SELECT COUNT(*) as count FROM {temp_table_name}"))$count
      print(glue("DEBUG - Records written to temp table: {temp_count}"))
      
      # Check how many would be considered duplicates
      dup_check <- dbGetQuery(con, glue("
        SELECT COUNT(*) as count
        FROM {temp_table_name} t
        INNER JOIN financial_data f 
        ON t.origin = f.origin AND t.series = f.series AND t.date = f.date
      "))$count
      print(glue("DEBUG - Records that already exist (duplicates): {dup_check}"))
      
      # Sample some records from temp and check if they exist in main table
      sample_records <- dbGetQuery(con, glue("
        SELECT t.origin, t.series, t.date, t.value,
               CASE WHEN f.origin IS NULL THEN 'NEW' ELSE 'EXISTS' END as status
        FROM {temp_table_name} t
        LEFT JOIN financial_data f 
        ON t.origin = f.origin AND t.series = f.series AND t.date = f.date
        LIMIT 5
      "))
      print("DEBUG - Sample records status:")
      print(sample_records)
    }
    
    # Insert only records that don't already exist
    insert_query <- glue("
      INSERT INTO financial_data (origin, series, date, value)
      SELECT t.origin, t.series, t.date, t.value
      FROM {temp_table_name} t
      LEFT JOIN financial_data f 
      ON t.origin = f.origin AND t.series = f.series AND t.date = f.date
      WHERE f.origin IS NULL
    ")
    
    # Execute the insert
    dbExecute(con, insert_query)
    
    # Get the count of inserted rows
    count_query <- glue("
      SELECT COUNT(*) as count
      FROM {temp_table_name} t
      LEFT JOIN financial_data f 
      ON t.origin = f.origin AND t.series = f.series AND t.date = f.date
      WHERE f.origin IS NULL
    ")
    
    rows_inserted <- dbGetQuery(con, count_query)$count
    
    # Clean up temporary table
    dbExecute(con, glue("DROP TABLE {temp_table_name}"))
    
    # If no rows were inserted, provide details about existing data
    if (rows_inserted == 0) {
      existing_info <- dbGetQuery(con, glue("
        SELECT 
          COUNT(*) as existing_count,
          MAX(date) as latest_date
        FROM financial_data 
        WHERE origin = '{origin}' AND series = '{series_name}'
      "))
      
      if (nrow(existing_info) > 0 && existing_info$existing_count > 0) {
        print(glue("Inserted 0 new rows for origin: {origin} ({series_name} - {existing_info$existing_count} existing records, latest: {existing_info$latest_date})"))
      } else {
        print(glue("Inserted 0 new rows for origin: {origin} ({series_name})"))
      }
    } else {
      print(glue("Inserted {rows_inserted} new rows for origin: {origin} ({series_name})"))
    }
    
    return(rows_inserted)
    
  }, error = function(e) {
    # Clean up temporary table in case of error
    tryCatch(dbExecute(con, glue("DROP TABLE IF EXISTS {temp_table_name}")), error = function(e2) {})
    print(glue("Error inserting data for origin {origin}: {e$message}"))
    return(0)
  })
}

# Function to calculate and insert ratios for tradeable assets
calculate_and_insert_ratios <- function(con) {
  print("\n=== CALCULATING RATIOS FOR TRADEABLE ASSETS ===")
  start_time <- Sys.time()
  
  # Define tradeable asset categories and patterns
  tradeable_patterns <- list(
    metals = c("GC=F", "SI=F", "PA=F"),  # Gold, Silver, Palladium
    crypto = c("BTC-USD", "ETH-USD"),
    oil = c("CL=F"),
    indices = c("^GSPC", "^IXIC", "^OMXC25", "^W5000"),
    bonds_long = c("TLT", "UBT", "IEF", "UST", "GOVT", "MBB"),
    bonds_short = c("SHV", "TBF", "TTT", "TBX", "PST", "TBT"),
    sector_etf = c("XLK", "XLV", "XLY", "XLP", "XLU", "XLF", "XLI", "XLB", "XLE", "XLRE", "XLC")
  )
  
  # Get all series that match tradeable patterns
  all_patterns <- unlist(tradeable_patterns)
  pattern_sql <- paste0("'", all_patterns, "'", collapse = ", ")
  
  tradeable_series <- dbGetQuery(con, glue("
    SELECT DISTINCT origin, series, MIN(date) as min_date, MAX(date) as max_date
    FROM financial_data
    WHERE series IN ({pattern_sql})
    GROUP BY origin, series
  "))
  
  if (nrow(tradeable_series) == 0) {
    print("No tradeable series found for ratio calculation")
    return(0)
  }
  
  print(glue("Found {nrow(tradeable_series)} tradeable series"))
  
  # Generate all pairwise combinations
  n_series <- nrow(tradeable_series)
  if (n_series < 2) {
    print("Need at least 2 series for ratios")
    return(0)
  }
  
  combinations <- combn(n_series, 2, simplify = FALSE)
  print(glue("Generating {length(combinations)} ratio combinations"))
  
  total_rows <- 0
  pb <- progress_bar$new(
    total = length(combinations),
    format = "[:bar] :percent :current/:total ETA: :eta"
  )
  
  for (combo in combinations) {
    pb$tick()
    
    idx1 <- combo[1]
    idx2 <- combo[2]
    
    series1 <- tradeable_series$series[idx1]
    series2 <- tradeable_series$series[idx2]
    origin1 <- tradeable_series$origin[idx1]
    origin2 <- tradeable_series$origin[idx2]
    
    tryCatch({
      # Get data for both series
      data1 <- dbGetQuery(con, glue("
        SELECT date, value as value1
        FROM financial_data
        WHERE origin = '{origin1}' AND series = '{series1}'
      "))
      
      data2 <- dbGetQuery(con, glue("
        SELECT date, value as value2
        FROM financial_data
        WHERE origin = '{origin2}' AND series = '{series2}'
      "))
      
      # Convert dates
      data1$date <- as.Date(data1$date)
      data2$date <- as.Date(data2$date)
      
      if (nrow(data1) == 0 || nrow(data2) == 0) next
      
      # Merge on date
      merged <- merge(data1, data2, by = "date", all = FALSE)
      
      if (nrow(merged) < 10) next  # Skip if too few overlapping dates
      
      # Calculate ratio
      merged$value <- merged$value1 / merged$value2
      
      # Remove infinite and NA values
      merged <- merged[is.finite(merged$value) & !is.na(merged$value), ]
      
      if (nrow(merged) == 0) next
      
      # Create ratio series name
      ratio_series <- paste0(series1, "_per_", series2)
      ratio_series <- clean_series_name(ratio_series)
      
      # Prepare data for insertion
      ratio_data <- data.frame(
        series = ratio_series,
        date = merged$date,
        value = merged$value,
        stringsAsFactors = FALSE
      )
      
      # Apply LOCF expansion
      ratio_data_expanded <- expand_to_daily_locf(ratio_data)
      
      if (nrow(ratio_data_expanded) > 0) {
        # Insert without additional LOCF (already applied)
        rows_inserted <- insert_data(con, ratio_data_expanded, "RATIO", apply_locf = FALSE)
        total_rows <- total_rows + rows_inserted
      }
      
    }, error = function(e) {
      # Silent fail, continue with other ratios
    })
  }
  
  end_time <- Sys.time()
  elapsed <- round(difftime(end_time, start_time, units = "mins"), 2)
  
  print(glue("\nRatio calculation completed in {elapsed} minutes"))
  print(glue("Total ratio rows inserted: {total_rows}"))
  
  # Summary of ratios created
  ratio_summary <- dbGetQuery(con, "
    SELECT COUNT(DISTINCT series) as num_ratios
    FROM financial_data
    WHERE origin = 'RATIO'
  ")
  
  if (nrow(ratio_summary) > 0) {
    print(glue("Total unique ratios in database: {ratio_summary$num_ratios}"))
  }
  
  return(total_rows)
}

# Function to clean and standardize series names
clean_series_name <- function(series_name) {
  series_name %>%
    str_replace_all("[^A-Za-z0-9_.-]", "_") %>%
    str_replace_all("_+", "_") %>%
    str_trim()
}

# Debug function to understand data flow issues
debug_data_collection <- function(con) {
  print("=== DEBUGGING DATA COLLECTION ===")
  
  # Check what data we have in the database for FRED
  existing_fred_data <- dbGetQuery(con, "
    SELECT series, MIN(date) as min_date, MAX(date) as max_date, COUNT(*) as count 
    FROM financial_data 
    WHERE origin = 'FRED' 
    GROUP BY series
  ")
  print("Existing FRED data in database:")
  print(existing_fred_data)
  
  # Check the latest date for FRED
  latest_fred_date <- get_latest_date(con, "FRED")
  print(glue("Latest FRED date in database: {latest_fred_date}"))
  
  # Get MORTGAGE30US data and see what we're filtering
  if (exists("fred_api_key") && fred_api_key != "") {
    series_data <- fredr::fredr_series_observations(series_id = "MORTGAGE30US")
    print(glue("Total MORTGAGE30US records from FRED: {nrow(series_data)}"))
    
    # Check date range of the data
    if (nrow(series_data) > 0) {
      print(glue("FRED data date range: {min(series_data$date)} to {max(series_data$date)}"))
      
      # See how many records are newer than latest_fred_date
      recent_data <- series_data %>%
        filter(date > latest_fred_date)
      print(glue("Records newer than {latest_fred_date}: {nrow(recent_data)}"))
      
      # Show last few records
      print("Last 5 FRED records:")
      print(tail(series_data[c("date", "value")], 5))
    }
  }
  
  # Check if the database has any data at all
  total_records <- dbGetQuery(con, "SELECT COUNT(*) as count FROM financial_data")$count
  print(glue("Total records in financial_data table: {total_records}"))
  
  # If database is empty, let's force insert some data to test
  if (total_records == 0 && exists("fred_api_key") && fred_api_key != "") {
    print("Database is empty - inserting test data...")
    
    series_data <- fredr::fredr_series_observations(series_id = "MORTGAGE30US")
    
    # Transform to harmonized format
    harmonized_data <- series_data %>%
      select(date, value) %>%
      mutate(
        series = "MORTGAGE30US",
        date = ymd(date),
        value = as.numeric(value)
      ) %>%
      filter(!is.na(value)) %>%
      slice_tail(n = 10)  # Just take last 10 records for testing
    
    print(glue("Attempting to insert {nrow(harmonized_data)} test records"))
    rows_inserted <- insert_data(con, harmonized_data, "FRED")
    
    # Verify insertion
    new_total <- dbGetQuery(con, "SELECT COUNT(*) as count FROM financial_data")$count
    print(glue("Records after test insertion: {new_total}"))
  }
}

print("Utility functions defined successfully!")


## API Configuration

In [ ]:
readLines(".env")

In [ ]:
# Load environment variables from .env file
env_path <- "datacollection/.env"

if (file.exists(env_path)) {
  # Read .env file and set environment variables
  env_lines <- readLines(env_path)
  for (line in env_lines) {
    if (nchar(line) > 0 && !startsWith(line, "#")) {
      parts <- strsplit(line, "=", fixed = TRUE)[[1]]
      if (length(parts) >= 2) {
        var_name <- trimws(parts[1])
        var_value <- trimws(paste(parts[-1], collapse = "="))  # Handle values with = in them
        do.call(Sys.setenv, setNames(list(var_value), var_name))
      }
    }
  }
  print(glue("Loaded environment from: {env_path}"))
} else {
  print(glue("Warning: .env file not found at {env_path}"))
}

# Set up FRED API
fred_api_key <- Sys.getenv("fred_api_key")
if (fred_api_key != "") {
  fredr::fredr_set_key(key = fred_api_key)
  print(glue("FRED API key configured successfully! (Key: {substr(fred_api_key, 1, 8)}...)"))
} else {
  print("Warning: FRED API key not found. Please set fred_api_key in .env file.")
}

print("API configuration completed!")

## 1. FRED API Data Collection

In [ ]:
tags <- c("usa", "nsa", "annual", "monthly", "sa")

# 1) Loop through all tags, call with one tag at a time
all_series <- map_dfr(tags, \(tag) {
  tryCatch({
    fredr_tags_series(
      tag_names = tag,
      order_by = "popularity",
      sort_order = "desc"
    )
  }, error = function(e) {
    warning(glue("Failed for tag '{tag}': {e$message}"))
    tibble()
  })
})

# 3) Group by id to find max popularity
# 4) Filter top 20 by popularity
# 5) Pull id
top_20_ids <- all_series %>%
  group_by(id) %>%
  summarise(popularity = max(popularity, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(popularity)) %>%
  head(50) %>%
  pull(id)

top_20_ids

In [ ]:
# Quick database check - what's already in the database?
print("=== CURRENT DATABASE CONTENTS ===")

total_records <- dbGetQuery(con, "SELECT COUNT(*) as count FROM financial_data")$count
print(glue("Total records in database: {total_records}"))

# Count by origin
origin_summary <- dbGetQuery(con, "
  SELECT origin, 
         COUNT(*) as record_count,
         MIN(date) as earliest_date,
         MAX(date) as latest_date,
         COUNT(DISTINCT series) as num_series
  FROM financial_data 
  GROUP BY origin 
  ORDER BY origin
")
print("\nSummary by origin:")
print(origin_summary)

# If you want to start fresh, you can clear the database:
# dbExecute(con, "DELETE FROM financial_data")
# print("Database cleared - ready for fresh data collection")

In [ ]:
collect_fred_data <- function(con) {
  print("Starting FRED data collection...")
  
  # Define FRED series IDs based on the existing notebooks
  fred_series <- unique(c(top_20_ids,
    # Interest Rates
    "T5YIFR",           # 5-Year, 5-Year Forward Inflation Expectation Rate
    "BAA10Y",           # Moody's Seasoned Baa Corporate Bond Yield Relative to Yield on 10-Year Treasury
    "DFF",              # Federal Funds Rate
    "MORTGAGE30US",     # 30-Year Fixed Rate Mortgage Average
    "T10YIE",           # 10-Year Breakeven Inflation Rate
    "DGS10", "DGS20", "DGS30", "DGS5", "DGS2", "DGS1", # Treasury rates
    
    # Economic Indicators
    "CPIAUCSL",         # Consumer Price Index for All Urban Consumers
    "PCEPI",            # Personal Consumption Expenditures: Chain-type Price Index
    "CORESTICKM159SFRBATL", # Sticky Price Consumer Price Index less Food and Energy
    "PAYEMS",           # All Employees, Total Nonfarm
    "CIVPART",          # Civilian Labor Force Participation Rate
    "UNRATE",           # Unemployment Rate
    "ICSA",             # Initial Claims
    "INDPRO",           # Industrial Production Index
    "GDP",              # Gross Domestic Product
    
    # Housing
    "ACTLISCOUUS",      # Housing Inventory: Active Listing Count
    "COMPUTSA",         # New Privately-Owned Housing Units Completed
    "HOUST1F",          # New Privately-Owned Housing Units Started: 1-Unit Structures
    "COMPU1USA",        # New Privately-Owned Housing Units Completed: 1-Unit Structures
    "PERMIT",           # New Privately-Owned Housing Units Authorized by Building Permits
    "PERMIT1",          # New Privately-Owned Housing Units Authorized by Building Permits: 1-Unit Structures
    "HOUST",            # Housing Starts: Total New Privately Owned
    "CSUSHPINSA",       # S&P/Case-Shiller U.S. National Home Price Index
    
    # Money Supply and Financial
    "UMCSENT",          # University of Michigan: Consumer Sentiment
    "M2V",              # Velocity of M2 Money Stock
    "PSAVERT",          # Personal Saving Rate
    "M1SL",             # M1 Money Stock
    "M2SL",             # M2 Money Stock
    "MDSP",             # Median Sales Price of Houses Sold
    "AUTHNOTT",         # Motor Vehicle Retail Sales: Heavy Weight Trucks
    "AUTHNOT1U",        # Motor Vehicle Retail Sales: Light Weight Trucks
    "WALCL",            # All Federal Reserve Banks: Total Assets
    "MSPUS"             # Median Sales Price of Houses Sold for the United States
  ))
  
  # Get the latest date for FRED data
  latest_date <- get_latest_date(con, "FRED")
  print(glue("Latest FRED data date in database: {latest_date}"))
  
  total_rows <- 0
  pb <- progress_bar$new(total = length(fred_series), format = "[:bar] :percent :current/:total ETA: :eta")
  
  # Debug: Process first series with detailed output
  debug_first <- TRUE
  
  for (series_id in fred_series) {
    pb$tick()
    
    tryCatch({
      # Get series data from FRED
      series_data <- fredr::fredr_series_observations(
        series_id = series_id)
      
      if (debug_first) {
        print(glue("\nDEBUG - First series: {series_id}"))
        print(glue("Rows from FRED API: {nrow(series_data)}"))
        print("Sample of raw data:")
        print(head(series_data[c("date", "value")], 3))
      }
      
      if (nrow(series_data) > 0) {
        # Transform to harmonized format
        harmonized_data <- series_data %>%
          select(date, value) %>%
          mutate(
            series = series_id,
            date = ymd(date),
            value = as.numeric(value)
          ) %>%
          filter(!is.na(value))
        
        if (debug_first) {
          print(glue("Rows after harmonization: {nrow(harmonized_data)}"))
          print("Sample of harmonized data:")
          print(head(harmonized_data, 3))
          print(glue("Column names: {paste(names(harmonized_data), collapse=', ')}"))
        }
        
        # Insert data (with debug for first series)
        rows_inserted <- insert_data(con, harmonized_data, "FRED", debug = debug_first)
        total_rows <- total_rows + rows_inserted
        
        debug_first <- FALSE
      }
    }, error = function(e) {
      print(glue("Warning: Failed to collect {series_id}: {e$message}"))
    })
    
    # Small delay to be respectful to the API
    Sys.sleep(0.1)
  }
  
  print(glue("FRED data collection completed. Total rows inserted: {total_rows}"))
  return(total_rows)
}

# Execute FRED data collection
if (exists("fred_api_key") && fred_api_key != "") {
  fred_rows <- collect_fred_data(con)
} else {
  print("Skipping FRED data collection - API key not available")
  fred_rows <- 0
}

## 2. Yahoo Finance Data Collection

### 2.1 Load symbols from NYSE and NASDAQ

In [ ]:
# Load NASDAQ symbols
nasdaq_symbols <- read.csv("datacollection/symbols_nasdaq.csv", 
                           stringsAsFactors = FALSE)
cat("NASDAQ symbols loaded:", nrow(nasdaq_symbols), "rows\n")

# Load NYSE symbols
nyse_symbols <- read.csv("datacollection/symbols_nyse.csv", 
                         stringsAsFactors = FALSE)
cat("NYSE symbols loaded:", nrow(nyse_symbols), "rows\n")

# Combine both datasets (rbind appends rows)
all_symbols <- rbind(nasdaq_symbols, nyse_symbols)
cat("\nCombined symbols:", nrow(all_symbols), "rows\n")

# Sort by market cap descending and extract top symbols
top_n <- 10  # Default to top 10, can be changed

# Convert Market.Cap to numeric (remove commas and convert)
all_symbols$Market.Cap.Numeric <- as.numeric(gsub(",", "", all_symbols$Market.Cap))

# Sort by market cap and get top symbols
top_symbols_df <- all_symbols[order(-all_symbols$Market.Cap.Numeric, na.last = TRUE), ]
top_symbols_df <- head(top_symbols_df, top_n)

# Extract symbols as vector
top_symbols <- top_symbols_df$Symbol

cat("\nTop", top_n, "symbols by market cap:\n")
print(top_symbols_df[, c("Symbol", "Name", "Market.Cap")])

cat("\nTop symbols vector:\n")
print(top_symbols)

### 2.2 Load data

In [ ]:
collect_yahoo_data <- function(con) {
  print("Starting Yahoo Finance data collection...")
  
  # Define Yahoo Finance symbols based on existing notebooks
  yahoo_symbols <- c(
    # Top stocks by market cap
    top_symbols,
    
    # Major Indices
    "^GSPC",           # S&P 500
    "^IXIC",           # NASDAQ
    "^OMXC25",         # OMX Copenhagen 25
    "^W5000",          # Wilshire 5000
    
    # Commodities
    "GC=F",            # Gold
    "SI=F",            # Silver
    "CL=F",            # Crude Oil
    "PA=F",            # Palladium
    
    # Cryptocurrencies
    "BTC-USD",         # Bitcoin
    "ETH-USD",         # Ethereum
    
    # Currency
    "DX-Y.NYB",        # US Dollar Index
    "USDDKK=X",        # USD/DKK
    "DKKUSD=X",        # DKK/USD
    
    # Sector ETFs - Long
    "XLK",             # Technology
    "XLV",             # Health Care
    "XLY",             # Consumer Discretionary
    "XLP",             # Consumer Staples
    "XLU",             # Utilities
    "XLF",             # Financials
    "XLI",             # Industrials
    "XLB",             # Materials
    "XLE",             # Energy
    "XLRE",            # Real Estate
    "XLC",             # Communication Services
    
    # Bond ETFs - Short
    "SHV",             # iShares Short Treasury Bond ETF
    "TBF",             # ProShares Short 20+ Yr Treasury
    "TTT",             # UltraPro Short 20+ Year Treasury
    "TBX",             # Short 7-10 Year Treasury
    "PST",             # UltraShort 7-10 Year Treasury
    "TBT",             # UltraShort 20+ Year Treasury
    
    # Bond ETFs - Long
    "MBB",             # iShares MBS ETF
    "GOVT",            # iShares U.S. Treasury Bond ETF
    "SGOV",            # iShares 0-3 Month Treasury Bond ETF
    "SHY",             # iShares 1-3 Year Treasury Bond ETF
    "IEI",             # iShares 3-7 Year Treasury Bond ETF
    "IEF",             # iShares 7-10 Year Treasury Bond ETF
    "UST",             # Ultra 7-10 Year Treasury
    "UBT",             # Ultra 20+ Year Treasury
    "TLT",             # iShares 20+ Year Treasury Bond ETF
    
    # Sector Shorts
    "REW",             # ProShares UltraShort Technology
    "RXD",             # ProShares UltraShort Health Care
    "SCC",             # ProShares UltraShort Consumer Services
    "SDP",             # ProShares UltraShort Utilities
    "SKF",             # ProShares UltraShort Financials
    "SIJ",             # ProShares UltraShort Industrials
    "SMN",             # ProShares UltraShort Basic Materials
    "DUG",             # ProShares UltraShort Oil & Gas
    "SRS"              # ProShares UltraShort Real Estate
  )
  
  # Get the latest date for Yahoo data
  latest_date <- get_latest_date(con, "YAHOO")
  print(glue("Latest Yahoo data date in database: {latest_date}"))
  
  total_rows <- 0
  pb <- progress_bar$new(total = length(yahoo_symbols), format = "[:bar] :percent :current/:total ETA: :eta")
  
  for (symbol in yahoo_symbols) {
    pb$tick()
    
    tryCatch({
      # Get symbol data from Yahoo Finance
      # Only fetch data from latest_date onwards
      symbol_data <- getSymbols(
        symbol, 
        env = NULL,
        auto.assign = FALSE
      )
      
      if (!is.null(symbol_data) && nrow(symbol_data) > 0) {
        # Convert to data frame and clean
        df_symbol <- as.data.frame(symbol_data) %>%
          mutate(date = as.Date(row.names(.))) %>%
          select(date, 1) %>%  # Select date and first column (Open price)
          setNames(c("date", "value")) %>%
          mutate(
            series = symbol,
            date = as.Date(date),
            value = as.numeric(value)
          ) %>%
          filter(!is.na(value))
        
        # Reset row names
        row.names(df_symbol) <- NULL
        
        # Insert data
        if (nrow(df_symbol) > 0) {
          rows_inserted <- insert_data(con, df_symbol, "YAHOO")
          total_rows <- total_rows + rows_inserted
        }
      }
    }, error = function(e) {
      print(glue("Warning: Failed to collect {symbol}: {e$message}"))
    })
    
    # Random delay between 1-20 seconds to be respectful to Yahoo Finance API
    delay <- runif(1, min = 1, max = 20)
    Sys.sleep(delay)
  }
  
  print(glue("Yahoo Finance data collection completed. Total rows inserted: {total_rows}"))
  return(total_rows)
}

yahoo_rows <- collect_yahoo_data(con)
# Execute Yahoo Finance data collection

## 3. Danmarks Statistik Data Collection

In [ ]:
# =============================================================================
# Danmarks Statistik: Metadata-driven, flexible data collection system
#
# How it works:
#   1. Define tables in `dst_table_config` — each entry specifies the table ID,
#      a description, query parameters, optional post-fetch filters, and which
#      columns should be concatenated to form the series name.
#   2. `fetch_dst_table()` calls dst_meta() to inspect the table, then calls
#      dst_get_data() with meta_data for efficiency.
#   3. `collect_dst_data()` iterates the config, harmonises every table to
#      (series, date, value) and inserts into the database.
#
# To add a new DST table, just append an entry to `dst_table_config`.
# =============================================================================

# --- Configuration ---------------------------------------------------------
# Each entry:
#   table       – DST table ID
#   description – human-readable label (for logging)
#   query       – named list of variable values passed to dst_get_data()
#                 Use "*" to fetch all values for a variable.
#   filters     – optional named list of post-fetch filters.
#                 Name = column, value = character vector of accepted values.
#   series_cols – character vector of column names concatenated (with the table
#                 ID prepended) to build the series name.  If NULL, only the
#                 table ID is used as the series name.
#   lang        – language for the API call (default "da")

dst_table_config <- list(
  list(
    table       = "BYGV80",
    description = "Building permits and construction",
    query       = list(BYGFASE = "*", ANVENDELSE = "*", Tid = "*"),
    filters     = list(ANVENDELSE = c("140 Etageboliger", "120 Parcelhuse")),
    series_cols = c("BYGFASE", "ANVENDELSE")
  ),
  list(
    table       = "BYG1",
    description = "Construction employment",
    query       = list(BRANCHE07 = "*", "SÆSON" = "*", ART = "*", Tid = "*"),
    filters     = list("SÆSON" = "Sæsonkorrigeret"),
    series_cols = c("BRANCHE07", "ART")
  ),
  list(
    table       = "PRIS90",
    description = "Housing price index",
    query       = list(ENHED = "*", BOLTYP = "*", Tid = "*"),
    filters     = list(ENHED = "Indeks"),
    series_cols = c("BOLTYP", "ENHED")
  ),
  list(
    table       = "BYG42",
    description = "Building cost index",
    query       = list(HINDEKS = "*", DINDEKS = "*", ART = "*", TAL = "*", Tid = "*"),
    filters     = list(
      DINDEKS = "Byggeomkostningsindeks i alt",
      TAL     = "Indeks",
      ART     = c("Materialer", "Arbejdsomkostninger")
    ),
    series_cols = c("HINDEKS", "DINDEKS", "ART")
  ),
  list(
    table       = "PRIS114",
    description = "Consumer price index",
    query       = list(VAREGR = "00 Nettoprisindeks i alt", ENHED = "Indeks", Tid = "*"),
    filters     = NULL,
    series_cols = NULL   # single series → uses table name only
  )
)

# --- Core fetch function ---------------------------------------------------
fetch_dst_table <- function(cfg, lang = "da") {
  table_id <- cfg$table
  
  # 1. Retrieve metadata so we can pass it to dst_get_data for efficiency
  meta <- dst_meta(table = table_id, lang = lang)
  
  # Log available variables from metadata
  var_info <- meta$variables
  print(glue("  Metadata for {table_id}: {meta$basics$text}"))
  print(glue("  Variables: {paste(var_info$id, collapse = ', ')}"))
  print(glue("  Mandatory (elimination=FALSE): {paste(var_info$id[!var_info$elimination], collapse = ', ')}"))
  
  # 2. Build the query — start from cfg$query
  query <- cfg$query
  
  # 3. Fetch data using meta_data for efficiency
  raw_data <- dst_get_data(
    table     = table_id,
    query     = query,
    lang      = lang,
    meta_data = meta,
    parse_dst_tid = FALSE
  )
  
  if (is.null(raw_data) || nrow(raw_data) == 0) {
    print(glue("  No data returned for {table_id}"))
    return(data.frame(series = character(), date = as.Date(character()), value = numeric()))
  }
  
  print(glue("  Fetched {nrow(raw_data)} rows from {table_id}"))
  
  # 4. Apply post-fetch filters
  if (!is.null(cfg$filters)) {
    for (col_name in names(cfg$filters)) {
      filter_vals <- cfg$filters[[col_name]]
      # Match on substring since DST prepends IDs (e.g. "140 Etageboliger")
      matching <- grepl(paste(filter_vals, collapse = "|"), raw_data[[col_name]], fixed = FALSE)
      raw_data <- raw_data[matching, ]
    }
    print(glue("  After filtering: {nrow(raw_data)} rows"))
  }
  
  if (nrow(raw_data) == 0) {
    return(data.frame(series = character(), date = as.Date(character()), value = numeric()))
  }
  
  # 5. Build series name from configured columns
  if (!is.null(cfg$series_cols) && length(cfg$series_cols) > 0) {
    series_parts <- lapply(cfg$series_cols, function(col) as.character(raw_data[[col]]))
    raw_data$series <- do.call(paste, c(list(table_id), series_parts, sep = "_"))
  } else {
    raw_data$series <- table_id
  }
  raw_data$series <- sapply(raw_data$series, clean_series_name)
  
  # 6. Parse the TID column to a proper date
  #    DST formats: "2024" (annual), "2024K1" (quarterly), "2024M01" (monthly)
  tid_col <- if ("TID" %in% names(raw_data)) "TID" else "Tid"
  tid_raw <- as.character(raw_data[[tid_col]])
  
  parse_dst_date <- function(x) {
    # Try POSIXct first (dst_get_data sometimes returns parsed dates)
    if (inherits(x, "POSIXct") || inherits(x, "Date")) return(as.Date(x))
    x <- as.character(x)
    # Quarterly: 2024K1 → 2024-01-01, 2024K2 → 2024-04-01, etc.
    if (grepl("K", x, fixed = TRUE)) {
      parts <- strsplit(x, "K")[[1]]
      yr <- as.integer(parts[1])
      qtr <- as.integer(parts[2])
      return(as.Date(paste0(yr, "-", sprintf("%02d", (qtr - 1) * 3 + 1), "-01")))
    }
    # Monthly: 2024M01 → 2024-01-01
    if (grepl("M", x, fixed = TRUE)) {
      parts <- strsplit(x, "M")[[1]]
      return(as.Date(paste0(parts[1], "-", sprintf("%02d", as.integer(parts[2])), "-01")))
    }
    # Annual: 2024 → 2024-01-01
    if (grepl("^[0-9]{4}$", x)) {
      return(as.Date(paste0(x, "-01-01")))
    }
    # Fallback: try direct parse
    return(as.Date(x))
  }
  
  raw_data$date <- as.Date(sapply(tid_raw, parse_dst_date), origin = "1970-01-01")
  raw_data$value <- as.numeric(raw_data$value)
  
  result <- raw_data[!is.na(raw_data$value), c("series", "date", "value")]
  print(glue("  Harmonised to {nrow(result)} rows, {length(unique(result$series))} series"))
  return(result)
}

# --- Main collection function -----------------------------------------------
collect_dst_data <- function(con, config = dst_table_config, lang = "da") {
  print("Starting Danmarks Statistik data collection...")
  
  latest_date <- get_latest_date(con, "DST")
  print(glue("Latest DST data date in database: {latest_date}"))
  
  total_rows <- 0
  
  for (cfg in config) {
    tryCatch({
      print(glue("\nCollecting {cfg$table} - {cfg$description}..."))
      
      tbl_data <- fetch_dst_table(cfg, lang = lang)
      
      if (nrow(tbl_data) > 0) {
        rows_inserted <- insert_data(con, tbl_data, "DST")
        total_rows <- total_rows + rows_inserted
      }
    }, error = function(e) {
      print(glue("Warning: Failed to collect {cfg$table}: {e$message}"))
    })
    
    Sys.sleep(0.5)  # Be respectful to the DST API
  }
  
  print(glue("\nDanmarks Statistik data collection completed. Total rows inserted: {total_rows}"))
  return(total_rows)
}

# Execute Danmarks Statistik data collection
dst_rows <- collect_dst_data(con)

## 4. FinansDanmark Data Collection

In [ ]:
collect_finansdanmark_data <- function(con) {
  print("Starting FinansDanmark data collection...")
  
  # Get the latest date for FinansDanmark data
  latest_date <- get_latest_date(con, "FINANSDANMARK")
  print(glue("Latest FinansDanmark data date in database: {latest_date}"))
  
  total_rows <- 0
  
  tryCatch({
    print("Collecting mortgage interest rates from FinansDanmark...")
    
    # Get current URL for the interest rate file
    source_url <- "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"
    
    page_content <- read_html(source_url)
    current_url <- page_content %>%
      html_node("body > main > div > div.page-header > div.page-header__content > div > div.row > div.col-12.col-md-8 > div > p:nth-child(9) > strong > span > a") %>%
      html_attr("href")
    
    if (!is.na(current_url)) {
      xlsx_url <- paste0("https://finansdanmark.dk/", current_url)
      
      # Download and read the Excel file
      temp_file <- tempfile(fileext = ".xlsx")
      download.file(xlsx_url, temp_file, mode = "wb", quiet = TRUE)
      
      df_interest <- read.xlsx(temp_file, startRow = 1)
      unlink(temp_file)
      
      # Process the interest rate data with improved date handling
      df_interest$År[1] <- 1997
      df_interest <- df_interest %>%
        fill(År, .direction = "down") %>%
        mutate(
          # Convert year and week to date - use a safer method
          # ISO week date: year, week number, and day 1 (Monday)
          Date = tryCatch({
            # Create a date string: "Year-Uge-1" where 1 is Monday
            date_str <- sprintf("%d-W%02d-1", År, Uge)
            # Use ISOdate to create the date
            ISOweek::ISOweek2date(date_str)
          }, error = function(e) {
            # If that fails, use a simpler approximation
            # First day of the year + (week * 7) days
            as.Date(paste(År, "01", "01", sep = "-")) + (Uge * 7)
          })
        ) %>%
        # Filter out any NA dates before calculating curve
        filter(!is.na(Date)) %>%
        mutate(
          CurveLongShort = as.numeric(Lang.rente) - as.numeric(Kort.rente)
        ) %>%
        select(Date, Kort.rente, Lang.rente, CurveLongShort) %>%
        gather(series, value, Kort.rente:CurveLongShort) %>%
        mutate(
          series = paste("FINANSDK", series, sep = "_"),
          date = as.Date(Date),
          value = as.numeric(value)
        ) %>%
        select(series, date, value) %>%
        # Filter out any remaining NA values
        filter(!is.na(value), !is.na(date))
      
      if (nrow(df_interest) > 0) {
        rows_inserted <- insert_data(con, df_interest, "FINANSDANMARK")
        total_rows <- total_rows + rows_inserted
      }
    } else {
      print("Warning: Could not find current URL for FinansDanmark interest rates")
    }
  }, error = function(e) {
    print(glue("Warning: Failed to collect FinansDanmark data: {e$message}"))
  })
  
  print(glue("FinansDanmark data collection completed. Total rows inserted: {total_rows}"))
  return(total_rows)
}

# Execute FinansDanmark data collection
finansdk_rows <- collect_finansdanmark_data(con)

## 5. ETF Data Collection

In [ ]:
collect_etf_data <- function(con) {
  print("Starting ETF data collection...")
  
  # Check if ETF list file exists (based on the existing notebooks)
  etf_file_paths <- c(
    "../Chapters/01_Indicators/etf_list.xlsx",
    "etf_list.xlsx",
    "data/etf_list.xlsx"
  )
  
  etf_file <- NULL
  for (path in etf_file_paths) {
    if (file.exists(path)) {
      etf_file <- path
      break
    }
  }
  
  if (is.null(etf_file)) {
    print("Warning: ETF list file not found. Skipping ETF data collection.")
    return(0)
  }
  
  # Get the latest date for ETF data
  latest_date <- get_latest_date(con, "ETF")
  print(glue("Latest ETF data date in database: {latest_date}"))
  
  total_rows <- 0
  
  tryCatch({
    # Read ETF list
    df_etfs <- read.xlsx(etf_file)
    
    if (!"Ticker" %in% names(df_etfs)) {
      print("Warning: 'Ticker' column not found in ETF file")
      return(0)
    }
    
    etf_tickers <- df_etfs$Ticker
    print(glue("Found {length(etf_tickers)} ETF tickers to collect"))
    
    pb <- progress_bar$new(total = length(etf_tickers), format = "[:bar] :percent :current/:total ETA: :eta")
    
    for (ticker in etf_tickers) {
      pb$tick()
      
      tryCatch({
        # Get ETF data from Yahoo Finance
        etf_data <- getSymbols(
          ticker, 
          env = NULL,
          auto.assign = FALSE
        )
        
        if (!is.null(etf_data) && nrow(etf_data) > 0) {
          # Convert to harmonized format
          df_etf <- as.data.frame(etf_data) %>%
            mutate(date = as.Date(row.names(.))) %>%
            select(date, 1) %>%  # Select date and first column (Open price)
            setNames(c("date", "value")) %>%
            mutate(
              series = paste("ETF", ticker, sep = "_"),
              date = as.Date(date),
              value = as.numeric(value)
            ) %>%
            select(series, date, value) %>%
            filter(!is.na(value))
          
          # Reset row names
          row.names(df_etf) <- NULL
          
          if (nrow(df_etf) > 0) {
            rows_inserted <- insert_data(con, df_etf, "ETF")
            total_rows <- total_rows + rows_inserted
          }
        }
      }, error = function(e) {
        print(glue("Warning: Failed to collect ETF {ticker}: {e$message}"))
      })
      
      # Random delay between 1-20 seconds to be respectful to Yahoo Finance API
      delay <- runif(1, min = 1, max = 20)
      Sys.sleep(delay)
    }
  }, error = function(e) {
    print(glue("Warning: Failed to process ETF data: {e$message}"))
  })
  
  print(glue("ETF data collection completed. Total rows inserted: {total_rows}"))
  return(total_rows)
}

# Execute ETF data collection
etf_rows <- collect_etf_data(con)

## 6. Housing Statistics Files Processing

In [ ]:
# =============================================================================
# DANISH PROPERTY STATISTICS (BOLIG) DATA COLLECTION
# =============================================================================
# Sources: FinansDanmark Boligstatistik Excel files
# Data includes:
#   - BM011: Property Prices (Kr/m2) by price type, property category, postal code (quarterly)
#   - BM021: Market Movements (Count) - sold/listed properties (quarterly)
#   - BM031: Sale Times (Days) by property category, postal code (quarterly)
#   - UL30:  Mortgage Lending (Billion DKK) by category, region, loan type (quarterly)
#   - UDB010: Properties on Market (Count) by property category, region (monthly)
#   - UDB020: Property Prices (Kr/m2) by price type, category, region (monthly)
#   - UDB030: Listing/Sale Times (Days) by property category, region (monthly)
# =============================================================================

# Install readxl if needed
if (!require(readxl, quietly = TRUE)) {
  install.packages("readxl")
  library(readxl)
}

# Helper function to parse Danish quarter format (e.g., "2024K1" -> Date)
parse_danish_quarter <- function(quarter_str) {
  year <- as.integer(substr(quarter_str, 1, 4))
  q <- as.integer(substr(quarter_str, 6, 6))
  month <- q * 3
  date <- as.Date(paste(year, month, "01", sep = "-"))
  ceiling_date(date, "month") - days(1)
}

# Helper function to parse Danish month format (e.g., "2024M01" -> Date)
parse_danish_month <- function(month_str) {
  year <- as.integer(substr(month_str, 1, 4))
  month <- as.integer(substr(month_str, 6, 7))
  date <- as.Date(paste(year, month, "01", sep = "-"))
  ceiling_date(date, "month") - days(1)
}

# Generic function to process FinansDanmark-style Excel files
process_finansdk_bolig_file <- function(file_path, table_code, n_header_cols, debug = FALSE) {
  
  cat("\n=== Processing", table_code, "===\n")
  
  df <- read_excel(file_path, col_names = FALSE)
  
  title <- as.character(df[1, 1])
  unit <- as.character(df[2, 1])
  
  cat("Title:", title, "\n")
  cat("Unit:", unit, "\n")
  
  # Find time periods (row 3)
  time_row <- 3
  time_periods <- as.character(df[time_row, (n_header_cols + 1):ncol(df)])
  time_periods <- time_periods[!is.na(time_periods)]
  
  is_quarterly <- grepl("K[1-4]$", time_periods[1])
  is_monthly <- grepl("M[0-9]{2}$", time_periods[1])
  
  cat("Time format:", ifelse(is_quarterly, "Quarterly", "Monthly"), 
      "| Range:", time_periods[1], "to", tail(time_periods, 1), "\n")
  
  # Data starts at row 4
  data_rows <- df[4:nrow(df), ]
  
  header_cols <- data_rows[, 1:n_header_cols]
  names(header_cols) <- paste0("COL", 1:n_header_cols)
  
  # Fill down NA values (hierarchical structure)
  for (i in 1:n_header_cols) {
    header_cols[[i]] <- zoo::na.locf(header_cols[[i]], na.rm = FALSE)
  }
  
  data_cols <- data_rows[, (n_header_cols + 1):ncol(data_rows)]
  names(data_cols) <- time_periods
  
  combined <- cbind(header_cols, data_cols)
  
  # Pivot to long format
  tidy_data <- combined %>%
    pivot_longer(
      cols = all_of(time_periods),
      names_to = "time_period",
      values_to = "value"
    ) %>%
    filter(!is.na(value) & value != ".." & value != "..") %>%
    mutate(
      value = as.numeric(value),
      date = if (is_quarterly) {
        sapply(time_period, parse_danish_quarter)
      } else {
        sapply(time_period, parse_danish_month)
      },
      date = as.Date(date, origin = "1970-01-01"),
      series = paste(table_code, 
                     sapply(across(starts_with("COL")), as.character) %>% 
                       apply(1, function(x) paste(x[!is.na(x)], collapse = "_")),
                     sep = "_"),
      series = clean_series_name(series)
    ) %>%
    filter(!is.na(value) & !is.na(date)) %>%
    select(series, date, value)
  
  cat("Processed rows:", nrow(tidy_data), "| Unique series:", n_distinct(tidy_data$series), "\n")
  
  return(tidy_data)
}

# File specifications: table_code -> n_header_cols
file_specs <- list(
  "BM011" = 3,  "BM021" = 3, "BM031" = 2,
  "UL30"  = 4,  "UDB010" = 3, "UDB020" = 3, "UDB030" = 3
)

# Main collection function
collect_danish_property_data <- function(con, folder_path) {
  print("Starting Danish Property Statistics (Bolig) data collection...")
  
  files <- list.files(folder_path, pattern = "\\.xlsx$", full.names = TRUE)
  if (length(files) == 0) {
    print(glue("No Excel files found in {folder_path}"))
    return(0)
  }
  
  print(glue("Found {length(files)} Excel files"))
  
  latest_date <- get_latest_date(con, "BOLIG_DK")
  print(glue("Latest BOLIG_DK date in database: {latest_date}"))
  
  total_rows <- 0
  
  for (file in files) {
    fname <- basename(file)
    table_code <- NULL
    for (code in names(file_specs)) {
      if (grepl(code, fname, ignore.case = TRUE)) {
        table_code <- code
        break
      }
    }
    
    if (is.null(table_code)) next
    
    tryCatch({
      tidy_data <- process_finansdk_bolig_file(file, table_code, file_specs[[table_code]])
      if (nrow(tidy_data) > 0) {
        tidy_data <- filter(tidy_data, date > latest_date)
        if (nrow(tidy_data) > 0) {
          rows_inserted <- insert_data(con, tidy_data, "BOLIG_DK", apply_locf = FALSE)
          total_rows <- total_rows + rows_inserted
        }
      }
    }, error = function(e) {
      print(glue("Error processing {fname}: {e$message}"))
    })
  }
  
  print(glue("\nDanish Property Statistics completed. Total rows inserted: {total_rows}"))
  return(total_rows)
}

cat("Danish property data functions defined!\n")

In [ ]:
# Execute Danish Property Statistics data collection
# Point to the folder containing the Excel files
bolig_folder <- "datacollection/Bolig/2026_02"

# Check what files are available
cat("=== Files in Bolig folder ===\n")
print(list.files(bolig_folder, pattern = "\\.xlsx$"))

# Collect data
bolig_rows <- collect_danish_property_data(con, bolig_folder)

In [ ]:
# =============================================================================
# VERIFY DANISH PROPERTY DATA IN DATABASE
# =============================================================================

bolig_summary <- dbGetQuery(con, "
  SELECT 
    origin,
    COUNT(*) as total_records,
    COUNT(DISTINCT series) as unique_series,
    MIN(date) as earliest_date,
    MAX(date) as latest_date
  FROM financial_data 
  WHERE origin = 'BOLIG_DK'
  GROUP BY origin
")

cat("=== BOLIG_DK Summary ===\n")
print(bolig_summary)

# Show series by table code
cat("\n=== Series Count by Table ===\n")
series_by_table <- dbGetQuery(con, "
  SELECT 
    CASE 
      WHEN series LIKE 'BM011%' THEN 'BM011 - Property Prices (quarterly)'
      WHEN series LIKE 'BM021%' THEN 'BM021 - Market Movements (quarterly)'
      WHEN series LIKE 'BM031%' THEN 'BM031 - Sale Times (quarterly)'
      WHEN series LIKE 'UL30%' THEN 'UL30 - Mortgage Lending (quarterly)'
      WHEN series LIKE 'UDB010%' THEN 'UDB010 - Properties on Market (monthly)'
      WHEN series LIKE 'UDB020%' THEN 'UDB020 - Property Prices (monthly)'
      WHEN series LIKE 'UDB030%' THEN 'UDB030 - Listing Times (monthly)'
      ELSE 'Other'
    END as table_name,
    COUNT(DISTINCT series) as n_series,
    COUNT(*) as n_records,
    MIN(date) as earliest,
    MAX(date) as latest
  FROM financial_data 
  WHERE origin = 'BOLIG_DK'
  GROUP BY 1
  ORDER BY 1
")
print(series_by_table)

# Sample recent data
cat("\n=== Sample Recent Data (Copenhagen Region, Jan 2026) ===\n")
sample_data <- dbGetQuery(con, "
  SELECT series, date, value 
  FROM financial_data 
  WHERE origin = 'BOLIG_DK' 
    AND date >= '2026-01-01'
    AND series LIKE '%Hovedstaden%'
  ORDER BY series, date DESC
  LIMIT 12
")
print(sample_data)

In [ ]:
collect_housing_files_data <- function(con) {
  print("Starting housing statistics files processing...")
  
  # Look for housing data files in various locations
  data_paths <- c(
    "../vignettes/Data/",
    "data/",
    "Data/",
    "../Data/"
  )
  
  housing_files <- c(
    "UDB010.xlsx",  # Housing supply
    "BM010.xlsx",   # Housing prices
    "UDB030.xlsx",  # Housing sale times
    "BM020.xlsx"    # Housing sales
  )
  
  # Get the latest date for housing files data
  latest_date <- get_latest_date(con, "HOUSING_FILES")
  print(glue("Latest housing files data date in database: {latest_date}"))
  
  total_rows <- 0
  
  for (file_name in housing_files) {
    file_found <- FALSE
    
    for (data_path in data_paths) {
      file_path <- paste0(data_path, file_name)
      
      if (file.exists(file_path)) {
        file_found <- TRUE
        print(glue("Processing {file_name} from {data_path}"))
        
        tryCatch({
          if (file_name == "UDB010.xlsx") {
            # Housing supply data
            df_housing <- read.xlsx(file_path, startRow = 3) %>%
              rename(
                SUPPLY_TYPE = X2,
                PROPERTY_TYPE = X1,
                ZIP = X3
              ) %>%
              fill(SUPPLY_TYPE, .direction = "down") %>%
              fill(PROPERTY_TYPE, .direction = "down") %>%
              fill(ZIP, .direction = "down") %>%
              gather(DATE, VALUE, 4:ncol(.)) %>%
              filter(VALUE != "..") %>%
              mutate(
                DATE = ceiling_date(ymd(glue("{substr(DATE,1,4)}-{substr(DATE,6,7)}-01")), unit="month") - days(1),
                series = paste("HOUSING_SUPPLY", PROPERTY_TYPE, SUPPLY_TYPE, ZIP, sep = "_"),
                series = clean_series_name(series),
                date = as.Date(DATE),
                value = as.numeric(VALUE)
              ) %>%
              select(series, date, value) %>%
              filter(!is.na(value), date > latest_date)
            
          } else if (file_name == "BM010.xlsx") {
            # Housing prices data
            df_housing <- read.xlsx(file_path, startRow = 3) %>%
              rename_with(~c("PROPERTY_TYPE", "SALES_TYPE", "ZIP"), 1:3) %>%
              fill(PROPERTY_TYPE, .direction = "down") %>%
              fill(SALES_TYPE, .direction = "down") %>%
              gather(DATE, VALUE, 4:ncol(.)) %>%
              filter(VALUE != "..") %>%
              mutate(
                DATE = ceiling_date(as.Date(as.yearqtr(DATE, format = "%YK%q")), "quarters") - days(1),
                series = paste("HOUSING_PRICES", PROPERTY_TYPE, SALES_TYPE, ZIP, sep = "_"),
                series = clean_series_name(series),
                date = as.Date(DATE),
                value = as.numeric(VALUE)
              ) %>%
              select(series, date, value) %>%
              filter(!is.na(value), date > latest_date)
            
          } else if (file_name == "UDB030.xlsx") {
            # Housing sale times data
            df_housing <- read.xlsx(file_path, startRow = 3) %>%
              rename(
                STAT = X2,
                PROPERTY_TYPE = X1,
                ZIP = X3
              ) %>%
              fill(STAT, .direction = "down") %>%
              fill(PROPERTY_TYPE, .direction = "down") %>%
              filter(STAT == "Liggetider (dage)") %>%
              gather(DATE, VALUE, 4:ncol(.)) %>%
              filter(VALUE != "..") %>%
              mutate(
                DATE = ceiling_date(ymd(glue("{substr(DATE,1,4)}-{substr(DATE,6,7)}-01")), unit="month") - days(1),
                series = paste("HOUSING_SALETIMES", PROPERTY_TYPE, ZIP, sep = "_"),
                series = clean_series_name(series),
                date = as.Date(DATE),
                value = as.numeric(VALUE)
              ) %>%
              select(series, date, value) %>%
              filter(!is.na(value), date > latest_date)
            
          } else if (file_name == "BM020.xlsx") {
            # Housing sales data
            df_housing <- read.xlsx(file_path, startRow = 3) %>%
              rename(
                SALES_TYPE = X2,
                PROPERTY_TYPE = X1,
                ZIP = X3
              ) %>%
              fill(SALES_TYPE, .direction = "down") %>%
              fill(PROPERTY_TYPE, .direction = "down") %>%
              gather(DATE, VALUE, 4:ncol(.)) %>%
              filter(VALUE != "..") %>%
              mutate(
                DATE = ceiling_date(as.Date(as.yearqtr(DATE, format = "%YK%q")), "quarters") - days(1),
                series = paste("HOUSING_SALES", PROPERTY_TYPE, SALES_TYPE, ZIP, sep = "_"),
                series = clean_series_name(series),
                date = as.Date(DATE),
                value = as.numeric(VALUE)
              ) %>%
              select(series, date, value) %>%
              filter(!is.na(value), date > latest_date)
          }
          
          if (exists("df_housing") && nrow(df_housing) > 0) {
            rows_inserted <- insert_data(con, df_housing, "HOUSING_FILES")
            total_rows <- total_rows + rows_inserted
          }
          
        }, error = function(e) {
          print(glue("Warning: Failed to process {file_name}: {e$message}"))
        })
        
        break  # Exit the path loop once file is found and processed
      }
    }
    
    if (!file_found) {
      print(glue("Warning: {file_name} not found in any of the data directories"))
    }
  }
  
  print(glue("Housing files data processing completed. Total rows inserted: {total_rows}"))
  return(total_rows)
}

# Execute housing files data collection
housing_files_rows <- collect_housing_files_data(con)

## Pipeline Summary and Data Validation

In [ ]:
# Mark current task as completed and start validation
print("\n=== ETL PIPELINE SUMMARY ===")
print(glue("FRED data rows inserted: {if(exists('fred_rows')) fred_rows else 0}"))
print(glue("Yahoo Finance rows inserted: {if(exists('yahoo_rows')) yahoo_rows else 0}"))
print(glue("Danmarks Statistik rows inserted: {if(exists('dst_rows')) dst_rows else 0}"))
print(glue("FinansDanmark rows inserted: {if(exists('finansdk_rows')) finansdk_rows else 0}"))
print(glue("ETF data rows inserted: {if(exists('etf_rows')) etf_rows else 0}"))
print(glue("Housing files rows inserted: {if(exists('housing_files_rows')) housing_files_rows else 0}"))

# Calculate ratios for tradeable assets
ratio_rows <- calculate_and_insert_ratios(con)
print(glue("Ratio data rows inserted: {ratio_rows}"))

total_rows_inserted <- sum(
  if(exists('fred_rows')) fred_rows else 0,
  if(exists('yahoo_rows')) yahoo_rows else 0,
  if(exists('dst_rows')) dst_rows else 0,
  if(exists('finansdk_rows')) finansdk_rows else 0,
  if(exists('etf_rows')) etf_rows else 0,
  if(exists('housing_files_rows')) housing_files_rows else 0,
  if(exists('ratio_rows')) ratio_rows else 0
)

print(glue("\nTOTAL ROWS INSERTED: {total_rows_inserted}"))
print("\n=== DATABASE VALIDATION ===")

In [ ]:
# Database validation queries
print("Database validation and summary:")

# Count total records
total_records <- dbGetQuery(con, "SELECT COUNT(*) as count FROM financial_data")$count
print(glue("Total records in database: {total_records}"))

# Count by origin
origin_counts <- dbGetQuery(con, "
  SELECT origin, COUNT(*) as count 
  FROM financial_data 
  GROUP BY origin 
  ORDER BY count DESC
")
print("\nRecords by origin:")
print(origin_counts)

# Date range by origin
date_ranges <- dbGetQuery(con, "
  SELECT origin, 
         MIN(date) as min_date, 
         MAX(date) as max_date,
         COUNT(DISTINCT series) as unique_series
  FROM financial_data 
  GROUP BY origin 
  ORDER BY origin
")
print("\nDate ranges and series count by origin:")
print(date_ranges)

# Recent data (last 30 days)
recent_data <- dbGetQuery(con, glue("
  SELECT origin, COUNT(*) as recent_count 
  FROM financial_data 
  WHERE date >= '{Sys.Date() - 30}'
  GROUP BY origin 
  ORDER BY recent_count DESC
"))
print("\nRecent data (last 30 days):")
print(recent_data)

# Sample of latest data
sample_data <- dbGetQuery(con, "
  SELECT origin, series, date, value 
  FROM financial_data 
  ORDER BY date DESC 
  LIMIT 10
")
print("\nSample of latest data:")
print(sample_data)

## Data Quality Checks

In [ ]:
print("\n=== DATA QUALITY CHECKS ===")

# Check for NULL values
null_check <- dbGetQuery(con, "
  SELECT 
    SUM(CASE WHEN origin IS NULL THEN 1 ELSE 0 END) as null_origins,
    SUM(CASE WHEN series IS NULL THEN 1 ELSE 0 END) as null_series,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) as null_dates,
    SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) as null_values
  FROM financial_data
")
print("NULL value counts:")
print(null_check)

# Check for duplicate records (using string concatenation for distinct count)
duplicate_check <- dbGetQuery(con, "
  SELECT COUNT(*) - COUNT(DISTINCT (origin || '|' || series || '|' || CAST(date AS VARCHAR))) as duplicates
  FROM financial_data
")$duplicates
print(glue("Duplicate records: {duplicate_check}"))

# Check data distribution by year
yearly_distribution <- dbGetQuery(con, "
  SELECT 
    strftime('%Y', date) as year,
    COUNT(*) as count
  FROM financial_data 
  GROUP BY strftime('%Y', date)
  ORDER BY year DESC
  LIMIT 10
")
print("\nData distribution by year (recent 10 years):")
print(yearly_distribution)

print("\n=== ETL PIPELINE COMPLETED SUCCESSFULLY ===")

## Database Connection Cleanup

In [ ]:
# Close database connection
dbDisconnect(con, shutdown = TRUE)
print("Database connection closed.")
print(glue("Database saved to: {DB_PATH}"))
print("\nETL pipeline execution completed!")

# Print final summary
cat("\n=== FINAL SUMMARY ===\n")
cat(glue("Database file: {DB_PATH}\n"))
cat(glue("Schema: ORIGIN - SERIES - DATE - VALUE\n"))
cat(glue("Total new records added: {total_rows_inserted}\n"))
cat("Delta loading implemented: Only new data since last run is collected\n")
cat("\nData sources successfully integrated:\n")
cat("  ✓ FRED API (Federal Reserve Economic Data)\n")
cat("  ✓ Yahoo Finance (Stocks, ETFs, Commodities)\n")
cat("  ✓ Danmarks Statistik (Danish Statistics)\n")
cat("  ✓ FinansDanmark (Danish Financial Industry)\n")
cat("  ✓ Housing statistics files (FinansDanmarks boligstatistik)\n")
cat("\nRun this notebook regularly to keep the database updated with latest data.\n")